# 02 — Descriptive & Diagnostic

*What happened, and why?* — the first two rungs of the analytics ladder, mirroring
`references/Day2_Team_Workbook.ipynb`.


Reusable logic belongs in `src/startup_outcomes/`, not here — this notebook imports it and
defines nothing. Strip all outputs before committing, and check that it still runs under
**Restart & Run All**.

In [ ]:
# Activates the inline backend so a returned Figure renders. plots.py itself never
# imports pyplot, so no global state is created by the library.
%matplotlib inline

from startup_outcomes import audit, descriptive, diagnostic, plots
from startup_outcomes.load import load_raw

df = load_raw(verify_checksum=True)
df.shape

## Part A — cleaning that changes the answer

The blanks in `AI_Adoption_Level` are the only missing data in the file. Two questions
follow: are they spread evenly, and does it matter how we handle them?

In [ ]:
for check, result in descriptive.run_all(df).items():
    print(f"{check}:\n  {result}\n")

### A2 · Is the missingness related to something?

`missingness_by_strata` gives the range; `diagnostic.missingness_permutation` decides
whether that range is more than chance produces. It is: the blanks depend on `Domain`
(p = 0.007), which rules out filling them with a global mode.

In [ ]:
blocks = descriptive.missingness_map_counts(df)
overall = descriptive.category_profile(df).loc["AI_Adoption_Level", "blank_rows"] / len(df) * 100
plots.missingness_map(blocks, overall_pct=float(overall))

### A3 · Five strategies, five answers

The workbooks' signature move. Here the spread is small — 0.46 pp — so the choice is
low-stakes, but it is still made explicitly rather than by default.

In [ ]:
strategies = descriptive.imputation_strategies(df)
print(strategies.to_string())
plots.imputation_sensitivity(strategies)

### A5 · Outliers — find them, then decide

Both conventional rules, because they disagree by an order of magnitude. The last field
is the decision: outliers concentrated in one category are a finding, not a data problem.

In [ ]:
for column in ("Total_Funding_USD_Millions", "Runway_Months_2024"):
    print(descriptive.outlier_report(df, column), "\n")
print(descriptive.duplicate_report(df))

## Part B — descriptive: what happened

In [ ]:
print(descriptive.numeric_profile(df).to_string())
print()
print(descriptive.category_profile(df).to_string())

### B3 · The summary table — the main descriptive deliverable

Every group statistic sits beside its `n`. That column is not decoration: a rate on forty
rows and a rate on four thousand are not the same claim.

In [ ]:
summary = descriptive.group_summary(df, "Runway_Months_2024", "Investor_Tier")
print(summary.to_string())

## Part C — six charts that earn their place

The last panel is the one people skip: rows per category, the visual form of the `n`
column.

In [ ]:
shape = descriptive.distribution_shape(df, "Total_Funding_USD_Millions")
plots.descriptive_grid(
    values=df["Total_Funding_USD_Millions"],
    mean=float(shape["mean"]),
    median=float(shape["median"]),
    summary=summary,
    frequencies=df["Investor_Tier"].value_counts(),
    correlations=df[[left for left, _ in audit.SIZE_PAIRS] + ["Valuation_USD_Millions"]].corr(),
    scatter_x=df["Revenue_ARR_Millions"],
    scatter_y=df["Valuation_USD_Millions"],
)

## Part D — diagnostic: why did it happen

Every effect below carries an interval or a permutation null. That is the one place this
project departs from the workbooks, and it is what lets "no effect" be a finding rather
than a shrug.

In [ ]:
findings = diagnostic.run_all(df)
for check, result in findings.items():
    print(f"{check}:\n  {result}\n")

### D2 · Correlation — and the two kinds

Pearson understates every size pair on raw values and agrees with Spearman once logged:
the relationships are monotone but curved. That is why these five columns are read as one
block rather than as five features.

In [ ]:
print(diagnostic.correlation_comparison(diagnostic.canonical_frame(df)).to_string())

### D5 · The reversal check (Simpson's paradox)

The most important cell in the workbook, and it pays off here. Overall, AI-native
companies close *less* often than non-adopters. Inside `Tier 2 VC` — 5,157 rows — the sign
flips.

In [ ]:
analysis = diagnostic.canonical_frame(df)
print(diagnostic.adoption_gap_by_tier(analysis).to_string())
plots.reversal_plot(diagnostic.reversal_table(analysis, "AI_Adoption_Level", "Investor_Tier"))

## Day 2 deliverable

1. **The pattern:** runway below six months raises the closed rate from 11.3% to 23.6% —
   a 12.33 pp step, 95% CI [11.14, 13.51].
2. **What does not move it:** `Domain` and `Country` spreads sit *below* the median spread
   that shuffling produces (p = 0.54 and 0.66). No sector or geographic signal exists.
3. **What moves it slightly:** AI adoption, 2.95 pp with CI [1.11, 5.01] — real, and far
   too small to act on. It reverses inside `Tier 2 VC`.
4. **Three explanations that would produce this pattern:** a hand-authored generator rule;
   a real survival threshold; reverse causation, where failing companies stop raising.
   **What the data cannot rule out:** all three. It has no randomisation and no instrument.
5. **The sentence never to write:** *X causes Y.*

## Scratch

Anything below that proves useful should be promoted into `src/` and covered by a test.